# Table 4 & 5 — Evidence-Grounded Reasoning Evaluation
**MSG-KG: 91 S&P Companies | Qwen2.5-7B (Generator) + Llama-3.1-8B (Judge)**

This notebook runs the full evaluation pipeline for Tables 4 (Reasoning Performance) and 5 (Ablation Study).

**Before running:** Upload `colab_eval_data.zip` (from `D:\PhD\Finance\MSGKG\colab_eval_data.zip`) to Colab.

**Runtime:** Set to **GPU** (T4) — Runtime → Change runtime type → T4 GPU

## Step 1: Setup — Install Ollama + Models

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh
print('Ollama installed.')

In [ ]:
# Start Ollama server in background
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print('Ollama server started.')

In [ ]:
# Pull both models (this takes ~5-10 min)
!ollama pull qwen2.5:7b
!ollama pull llama3.1:8b
print('\nBoth models ready.')

## Step 2: Upload & Extract Data

In [ ]:
# Upload colab_eval_data.zip
from google.colab import files
print('Select colab_eval_data.zip from your computer...')
uploaded = files.upload()
print(f'Uploaded: {list(uploaded.keys())}')

In [ ]:
# Extract data
import zipfile, os

WORK_DIR = '/content/msgkg_eval'
os.makedirs(WORK_DIR, exist_ok=True)

with zipfile.ZipFile('colab_eval_data.zip', 'r') as zf:
    zf.extractall(WORK_DIR)

# Verify
import pathlib
reg = pathlib.Path(WORK_DIR) / 'companies_registry.json'
sec_dir = pathlib.Path(WORK_DIR) / 'sec_data'
onto_dir = pathlib.Path(WORK_DIR) / 'ontology'

print(f'Registry: {reg.exists()}')
print(f'10-K files: {len(list(sec_dir.glob("*.txt")))}')
print(f'Ontology files: {len(list(onto_dir.glob("*.ttl")))}')

In [ ]:
# Install dependencies
!pip install -q sentence-transformers scikit-learn numpy tqdm python-docx

## Step 3: Evaluation Pipeline

In [ ]:
import json, re, pathlib, time, sys, os
import urllib.request
import numpy as np
from collections import defaultdict
from tqdm.notebook import tqdm
from datetime import datetime

# ── Config ──
BASE_DIR    = pathlib.Path(WORK_DIR)
REG_PATH    = BASE_DIR / 'companies_registry.json'
SEC_DIR     = BASE_DIR / 'sec_data'
ONTO_DIR    = BASE_DIR / 'ontology'
RESULTS_DIR = BASE_DIR / 'eval_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OLLAMA_URL      = 'http://localhost:11434'
GENERATOR_MODEL = 'qwen2.5:7b'
JUDGE_MODEL     = 'llama3.1:8b'
N_BOOTSTRAP     = 1000
CONFIDENCE      = 0.95

print(f'Config ready. Results dir: {RESULTS_DIR}')

In [ ]:
# ── Ollama helpers ──

def ollama_chat(model, messages, max_tokens=400, json_mode=False):
    payload = {
        'model': model, 'messages': messages, 'stream': False,
        'options': {'temperature': 0.1, 'num_predict': max_tokens},
    }
    if json_mode:
        payload['format'] = 'json'
    data = json.dumps(payload).encode('utf-8')
    req = urllib.request.Request(
        f'{OLLAMA_URL}/api/chat', data=data,
        headers={'Content-Type': 'application/json'},
    )
    try:
        with urllib.request.urlopen(req, timeout=180) as resp:
            result = json.loads(resp.read().decode('utf-8'))
            return result['message']['content'].strip()
    except Exception as e:
        print(f'  LLM error: {str(e)[:60]}')
        return ''

def parse_json_response(text):
    start = text.find('{')
    end = text.rfind('}') + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None

def swap_to_model(target, other):
    print(f'  Swapping to {target}...')
    payload = json.dumps({'model': other, 'prompt': '', 'stream': False, 'keep_alive': 0}).encode('utf-8')
    try:
        req = urllib.request.Request(f'{OLLAMA_URL}/api/generate', data=payload, headers={'Content-Type': 'application/json'})
        urllib.request.urlopen(req, timeout=30)
    except: pass
    time.sleep(2)
    # Load target
    payload = json.dumps({'model': target, 'prompt': 'hi', 'stream': False, 'options': {'num_predict': 5}}).encode('utf-8')
    try:
        req = urllib.request.Request(f'{OLLAMA_URL}/api/generate', data=payload, headers={'Content-Type': 'application/json'})
        urllib.request.urlopen(req, timeout=120)
    except: pass
    print(f'  {target} loaded.')

# Test connection
test = ollama_chat(GENERATOR_MODEL, [{'role': 'user', 'content': 'Say hi'}], max_tokens=10)
print(f'Ollama test: "{test[:30]}"')
print('OK!' if test else 'ERROR — check Ollama')

In [ ]:
# ── Data loading ──

def load_registry():
    with open(REG_PATH, encoding='utf-8') as f:
        return json.load(f)

def extract_item1(filepath):
    text = filepath.read_text(encoding='utf-8', errors='ignore')
    hdr = text.find('</SEC-HEADER>')
    if hdr > 0: text = text[hdr + len('</SEC-HEADER>'):]
    text = re.sub(r'\s+', ' ', text)
    m = re.search(r'(?:ITEM\s*1\.?\s*(?:BUSINESS|Business))', text, re.IGNORECASE)
    start = m.end() if m else 0
    em = re.search(r'(?:ITEM\s*1A\.?\s|ITEM\s*2\.?\s|PART\s*II\b)', text[start:], re.IGNORECASE)
    end = start + em.start() if em else min(start + 200000, len(text))
    return text[start:end].strip()[:100000]

def find_10k_file(cik):
    cik_stripped = cik.lstrip('0')
    for f in SEC_DIR.glob('cleaned_10-K_*.txt'):
        acc = f.stem.replace('cleaned_10-K_', '')
        file_cik = acc.split('-')[0].lstrip('0')
        if file_cik == cik_stripped:
            return f
    return None

def chunk_text(text, chunk_size=500, overlap=80):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk_words = words[i:i + chunk_size]
        if len(chunk_words) < 30: break
        chunks.append(' '.join(chunk_words))
        i += chunk_size - overlap
    return chunks

import unicodedata
def slug(name):
    s = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode()
    s = re.sub(r'[^\w\s-]', '', s).strip().lower()
    return re.sub(r'[-\s]+', '_', s)[:50]

def parse_ontology_ttl(cik, company_name):
    ttl_path = ONTO_DIR / f'{slug(company_name)}.ttl'
    if not ttl_path.exists():
        return {'mission': '', 'stakeholders': [], 'values': [], 'objectives': [], 'capabilities': [], 'domains': []}
    content = ttl_path.read_text(encoding='utf-8', errors='ignore')
    result = {'mission': '', 'stakeholders': [], 'values': [], 'objectives': [], 'capabilities': [], 'domains': []}
    m = re.search(r'msg:missionText\s+"([^"]*)"', content)
    if m: result['mission'] = m.group(1)
    type_map = {'Stakeholder': 'stakeholders', 'CorporateValue': 'values', 'CorporateObjective': 'objectives', 'BusinessCapability': 'capabilities', 'BusinessDomain': 'domains'}
    for line in content.split('\n'):
        for rdf_type, key in type_map.items():
            if f'a msg:{rdf_type}' in line or f'a fibo-be:{rdf_type}' in line:
                inst_match = re.match(r'(inst:\S+)', line)
                if inst_match:
                    inst_name = inst_match.group(1).replace('inst:', '')
                    clean = re.sub(r'_', ' ', inst_name.split('_', 1)[-1] if '_' in inst_name else inst_name)
                    result[key].append(clean)
    for m in re.finditer(r'rdfs:label\s+"([^"]*)"', content):
        label = m.group(1)
        pos = m.start()
        context = content[max(0, pos - 200):pos]
        for rdf_type, key in type_map.items():
            if rdf_type in context:
                if label not in result[key]: result[key].append(label)
                break
    return result

# Load everything
registry = load_registry()
print(f'Companies: {len(registry)}')

company_data = {}
for cik, info in tqdm(registry.items(), desc='Loading data'):
    name = info['name']
    ov = info.get('overview', {})
    mission = ov.get('mission', '')
    filepath = find_10k_file(cik)
    chunks = []
    if filepath:
        item1 = extract_item1(filepath)
        if len(item1) > 100: chunks = chunk_text(item1)
    kg_struct = parse_ontology_ttl(cik, name)
    company_data[cik] = {'name': name, 'sector': info.get('sector', ''), 'mission': mission, 'chunks': chunks, 'kg_struct': kg_struct}

print(f'With chunks: {sum(1 for c in company_data.values() if c["chunks"])}')
print(f'With KG: {sum(1 for c in company_data.values() if c["kg_struct"]["mission"])}')

In [ ]:
# ── Question generation ──

QUESTION_TEMPLATES = [
    "What is {company}'s stated mission, and what specific evidence from their 10-K filing supports it?",
    "How does {company}'s mission connect to their strategic priorities and operational capabilities?",
    "Trace how {company}'s mission is operationalized through specific strategic initiatives and mechanisms described in their 10-K filing.",
]

def generate_questions(registry):
    questions = []
    for cik, info in registry.items():
        name = info['name']
        for i, template in enumerate(QUESTION_TEMPLATES):
            questions.append({
                'qid': f'{cik}_q{i+1}', 'cik': cik, 'company': name,
                'sector': info.get('sector', 'Unknown'),
                'question': template.format(company=name),
                'q_type': ['factual', 'strategic', 'trace'][i],
            })
    return questions

questions = generate_questions(registry)
print(f'Total questions: {len(questions)} ({len(registry)} companies x 3)')

In [ ]:
# ── Retrieval engines ──

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def tfidf_retrieve(query, chunks, top_k=5):
    if not chunks: return []
    corpus = chunks + [query]
    vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
    tfidf = vectorizer.fit_transform(corpus)
    sims = cosine_similarity(tfidf[-1], tfidf[:-1]).flatten()
    top_idx = sims.argsort()[::-1][:top_k]
    return [chunks[i] for i in top_idx if sims[i] > 0]

_embedder = None
def get_embedder():
    global _embedder
    if _embedder is None:
        from sentence_transformers import SentenceTransformer
        _embedder = SentenceTransformer('all-MiniLM-L6-v2')
    return _embedder

def semantic_retrieve(query, chunks, top_k=5):
    if not chunks: return []
    model = get_embedder()
    query_emb = model.encode([query], normalize_embeddings=True)
    chunk_embs = model.encode(chunks, normalize_embeddings=True, show_progress_bar=False)
    sims = np.dot(chunk_embs, query_emb.T).flatten()
    top_idx = sims.argsort()[::-1][:top_k]
    return [chunks[i] for i in top_idx if sims[i] > 0]

# Pre-load embedder
_ = get_embedder()
print('Embedder loaded.')

In [ ]:
# ── 5 Reasoning Systems ──

def system_text_only(question, **kw):
    return ollama_chat(GENERATOR_MODEL, [
        {'role': 'system', 'content': 'You are a financial analyst. Answer questions about corporate strategy based on your knowledge.'},
        {'role': 'user', 'content': question},
    ], max_tokens=300)

def system_vector_retrieval(question, chunks, **kw):
    retrieved = tfidf_retrieve(question, chunks, top_k=5)
    context = '\n\n'.join(retrieved[:5])
    return ollama_chat(GENERATOR_MODEL, [
        {'role': 'system', 'content': 'You are a financial analyst. Answer based ONLY on the provided evidence passages. Cite specific passages.'},
        {'role': 'user', 'content': f'Evidence passages from 10-K filing:\n{context[:3000]}\n\nQuestion: {question}'},
    ], max_tokens=300)

def system_rag(question, chunks, **kw):
    retrieved = semantic_retrieve(question, chunks, top_k=5)
    context = '\n\n'.join(retrieved[:5])
    return ollama_chat(GENERATOR_MODEL, [
        {'role': 'system', 'content': 'You are a financial analyst. Answer based ONLY on the provided evidence passages. Cite specific passages.'},
        {'role': 'user', 'content': f'Retrieved passages from 10-K filing:\n{context[:3000]}\n\nQuestion: {question}'},
    ], max_tokens=300)

def system_graphrag(question, chunks, kg_struct, **kw):
    kg_lines = []
    if kg_struct.get('mission'): kg_lines.append(f"Mission: {kg_struct['mission']}")
    for s in kg_struct.get('stakeholders', []): kg_lines.append(f'Mission -> serves -> {s}')
    for v in kg_struct.get('values', []): kg_lines.append(f'Mission -> embodies -> {v}')
    for o in kg_struct.get('objectives', []): kg_lines.append(f'Mission -> pursues -> {o}')
    for c in kg_struct.get('capabilities', []): kg_lines.append(f'Strategy -> leverages -> {c}')
    for d in kg_struct.get('domains', []): kg_lines.append(f'Company -> operates in -> {d}')
    kg_context = '\n'.join(kg_lines) if kg_lines else 'No knowledge graph available.'
    retrieved = semantic_retrieve(question, chunks, top_k=3)
    text_context = '\n\n'.join(retrieved[:3])
    return ollama_chat(GENERATOR_MODEL, [
        {'role': 'system', 'content': 'You are a financial analyst. Use the knowledge graph triples AND text evidence to answer. Cite both.'},
        {'role': 'user', 'content': f'Knowledge Graph:\n{kg_context}\n\nText Evidence:\n{text_context[:2000]}\n\nQuestion: {question}'},
    ], max_tokens=400)

def system_msgkg(question, chunks, kg_struct, mission, **kw):
    path_lines = []
    if mission: path_lines.append(f'MISSION: {mission}')
    pillars = kg_struct.get('values', []) + kg_struct.get('objectives', [])
    if pillars:
        path_lines.append(f"STRATEGIC PILLARS: {', '.join(pillars[:5])}")
        for p in pillars[:3]: path_lines.append(f'  Mission -> {p}')
    mechanisms = kg_struct.get('capabilities', [])
    if mechanisms:
        path_lines.append(f"OPERATIONAL MECHANISMS: {', '.join(mechanisms[:5])}")
        for m in mechanisms[:3]:
            if pillars: path_lines.append(f'  {pillars[0]} -> {m}')
    reasoning_path = '\n'.join(path_lines) if path_lines else 'No reasoning path available.'
    retrieved = semantic_retrieve(question, chunks, top_k=5)
    evidence_text = '\n\n'.join(f'[Evidence {i+1}]: {r[:300]}' for i, r in enumerate(retrieved[:5]))
    kg_lines = []
    for s in kg_struct.get('stakeholders', []): kg_lines.append(f'serves_stakeholder({s})')
    for v in kg_struct.get('values', []): kg_lines.append(f'embodies_value({v})')
    for o in kg_struct.get('objectives', []): kg_lines.append(f'pursues_objective({o})')
    for c in kg_struct.get('capabilities', []): kg_lines.append(f'leverages_capability({c})')
    kg_triples = '; '.join(kg_lines) if kg_lines else 'N/A'
    return ollama_chat(GENERATOR_MODEL, [
        {'role': 'system', 'content': 'You are a financial analyst performing evidence-grounded reasoning. Use the structured reasoning path, knowledge graph, AND textual evidence to answer. Explicitly trace the Mission -> Strategic Pillar -> Operational Mechanism chain. Cite specific evidence.'},
        {'role': 'user', 'content': f'Mission-Strategy Reasoning Path:\n{reasoning_path}\n\nKG Relations: {kg_triples}\n\nEvidence from 10-K Filing:\n{evidence_text[:2500]}\n\nQuestion: {question}\n\nAnswer with explicit reasoning chain (Mission -> Pillar -> Mechanism) and cite evidence:'},
    ], max_tokens=500)

# Ablation variants
def ablation_no_graph(question, chunks, kg_struct, mission, **kw):
    retrieved = semantic_retrieve(question, chunks, top_k=5)
    evidence_text = '\n\n'.join(f'[Evidence {i+1}]: {r[:300]}' for i, r in enumerate(retrieved[:5]))
    return ollama_chat(GENERATOR_MODEL, [
        {'role': 'system', 'content': 'You are a financial analyst. Answer based on the mission statement and textual evidence. Cite evidence.'},
        {'role': 'user', 'content': f'Mission: {mission}\n\nEvidence:\n{evidence_text[:3000]}\n\nQuestion: {question}'},
    ], max_tokens=400)

def ablation_no_evidence(question, chunks, kg_struct, mission, **kw):
    path_lines = [f'MISSION: {mission}']
    pillars = kg_struct.get('values', []) + kg_struct.get('objectives', [])
    mechanisms = kg_struct.get('capabilities', [])
    for p in pillars[:5]: path_lines.append(f'  Mission -> {p}')
    for m in mechanisms[:5]:
        if pillars: path_lines.append(f'  {pillars[0]} -> {m}')
    return ollama_chat(GENERATOR_MODEL, [
        {'role': 'system', 'content': 'You are a financial analyst. Use the structured reasoning path to answer. Trace the Mission -> Pillar -> Mechanism chain.'},
        {'role': 'user', 'content': f'Reasoning Path:\n{chr(10).join(path_lines)}\n\nQuestion: {question}'},
    ], max_tokens=400)

def ablation_no_kg_structure(question, chunks, kg_struct, mission, **kw):
    retrieved = semantic_retrieve(question, chunks, top_k=5)
    evidence_text = '\n\n'.join(retrieved[:5])
    elements = []
    for key in ['stakeholders', 'values', 'objectives', 'capabilities', 'domains']:
        elements.extend(kg_struct.get(key, []))
    flat_kg = ', '.join(elements) if elements else 'N/A'
    return ollama_chat(GENERATOR_MODEL, [
        {'role': 'system', 'content': 'You are a financial analyst. Answer based on the retrieved text and related concepts. Cite evidence.'},
        {'role': 'user', 'content': f'Mission: {mission}\nRelated concepts: {flat_kg}\n\nText:\n{evidence_text[:3000]}\n\nQuestion: {question}'},
    ], max_tokens=400)

print('All 8 reasoning systems defined.')

In [ ]:
# ── LLM-as-Judge rubrics ──

GROUNDING_RUBRIC = """Score the answer's evidence grounding on a scale of 1-5:
5: All claims are supported by specific, verifiable evidence from the filing
4: Most claims supported, minor unsupported assertions
3: Some evidence cited but significant claims lack support
2: Minimal evidence, mostly unsupported claims
1: No evidence grounding, entirely generic or hallucinated

COMPANY: {company}
SECTOR: {sector}
REFERENCE MISSION: {mission}

QUESTION: {question}
ANSWER TO EVALUATE: {answer}

Respond with ONLY JSON: {{"grounding_score": N, "reason": "one sentence"}}"""

EXPLANATION_RUBRIC = """Score the answer's structural reasoning quality on a scale of 1-5:
5: Clear Mission -> Strategic Pillar -> Operational Mechanism chain with specific evidence
4: Mostly complete chain, minor gaps in reasoning
3: Partial chain, some strategic elements connected but incomplete
2: Weak structural reasoning, mostly flat description
1: No structural reasoning, just generic statements

COMPANY: {company}
COMPANY MISSION: {mission}
KNOWN STRATEGIC ELEMENTS: {strategic_elements}

QUESTION: {question}
ANSWER TO EVALUATE: {answer}

Respond with ONLY JSON: {{"explanation_score": N, "reason": "one sentence"}}"""

def judge_answer(question_data, answer, mission, kg_struct):
    company = question_data['company']
    sector = question_data['sector']
    question = question_data['question']
    strategic_elements = ', '.join(
        kg_struct.get('values', [])[:3] + kg_struct.get('objectives', [])[:3] + kg_struct.get('capabilities', [])[:3]
    )
    g_resp = ollama_chat(JUDGE_MODEL, [
        {'role': 'system', 'content': 'You are an expert evaluator. Score answers strictly. Respond with JSON only.'},
        {'role': 'user', 'content': GROUNDING_RUBRIC.format(company=company, sector=sector, mission=mission[:200], question=question, answer=answer[:500])},
    ], max_tokens=100, json_mode=True)
    g_json = parse_json_response(g_resp)
    g_score = g_json.get('grounding_score', 2) if g_json else 2
    e_resp = ollama_chat(JUDGE_MODEL, [
        {'role': 'system', 'content': 'You are an expert evaluator. Score answers strictly. Respond with JSON only.'},
        {'role': 'user', 'content': EXPLANATION_RUBRIC.format(company=company, mission=mission[:200], strategic_elements=strategic_elements[:300], question=question, answer=answer[:500])},
    ], max_tokens=100, json_mode=True)
    e_json = parse_json_response(e_resp)
    e_score = e_json.get('explanation_score', 2) if e_json else 2
    return max(1, min(5, int(g_score))), max(1, min(5, int(e_score)))

print('Judge rubrics ready.')

In [ ]:
# ── Statistical helpers ──

def normalize_score(raw_scores):
    return [(s - 1) / 4 * 100 for s in raw_scores]

def bootstrap_ci(scores, n_bootstrap=N_BOOTSTRAP, ci=CONFIDENCE):
    scores = np.array(scores)
    means = sorted([np.mean(np.random.choice(scores, size=len(scores), replace=True)) for _ in range(n_bootstrap)])
    alpha = (1 - ci) / 2
    return float(np.mean(scores)), float(means[int(alpha * n_bootstrap)]), float(means[int((1 - alpha) * n_bootstrap)])

def paired_bootstrap_test(scores_a, scores_b, n_bootstrap=N_BOOTSTRAP):
    scores_a, scores_b = np.array(scores_a), np.array(scores_b)
    count = sum(1 for _ in range(n_bootstrap) if np.mean(scores_a[np.random.choice(len(scores_a), len(scores_a), replace=True)]) - np.mean(scores_b[np.random.choice(len(scores_b), len(scores_b), replace=True)]) <= 0)
    return count / n_bootstrap

print('Stats ready.')

## Step 4: Run Generation (Qwen 7B) — ~1.5 hrs

In [ ]:
# ── PHASE 2: Generate answers with all 8 systems ──

SYSTEMS = {
    'Text-Only LLM': system_text_only,
    'Vector Retrieval + LLM': system_vector_retrieval,
    'SentenceTransformer + RAG': system_rag,
    'GraphRAG': system_graphrag,
    'MSG-KG Reasoning': system_msgkg,
}
ABLATIONS = {
    'Full MSG-KG Framework': system_msgkg,
    'Without Graph Retrieval': ablation_no_graph,
    'Without Evidence Linking': ablation_no_evidence,
    'Without KG Structure': ablation_no_kg_structure,
}
ALL_SYSTEMS = {**SYSTEMS, **ABLATIONS}

print(f'Systems: {len(ALL_SYSTEMS)} | Questions: {len(questions)}')
print(f'Total generation calls: ~{len(questions) * len(ALL_SYSTEMS)}')

swap_to_model(GENERATOR_MODEL, JUDGE_MODEL)

# Load checkpoint
checkpoint_path = RESULTS_DIR / 'checkpoint_answers.json'
all_answers = {}
if checkpoint_path.exists():
    with open(checkpoint_path, encoding='utf-8') as f:
        all_answers = json.load(f)
    print(f'Resuming: {len(all_answers)} questions done')

for qi, q in enumerate(tqdm(questions, desc='Generating answers')):
    qid = q['qid']
    cik = q['cik']
    cd = company_data[cik]

    if qid in all_answers and len(all_answers[qid]) >= len(ALL_SYSTEMS):
        continue
    if qid not in all_answers:
        all_answers[qid] = {}

    for sys_name, sys_func in ALL_SYSTEMS.items():
        if sys_name in all_answers[qid]: continue
        if sys_name == 'Full MSG-KG Framework' and 'MSG-KG Reasoning' in all_answers[qid]:
            all_answers[qid][sys_name] = all_answers[qid]['MSG-KG Reasoning']
            continue
        answer = sys_func(question=q['question'], chunks=cd['chunks'], kg_struct=cd['kg_struct'], mission=cd['mission'])
        all_answers[qid][sys_name] = answer or ''

    # Save every 5 questions
    if (qi + 1) % 5 == 0:
        with open(checkpoint_path, 'w', encoding='utf-8') as f:
            json.dump(all_answers, f, ensure_ascii=False)

with open(checkpoint_path, 'w', encoding='utf-8') as f:
    json.dump(all_answers, f, ensure_ascii=False)
print(f'Generation complete. Total answers: {sum(len(v) for v in all_answers.values())}')

## Step 5: Run Judging (Llama 3.1 8B) — ~1.5 hrs

In [ ]:
# ── PHASE 3: Judge all answers ──

swap_to_model(JUDGE_MODEL, GENERATOR_MODEL)

all_scores = {}
scores_checkpoint = RESULTS_DIR / 'checkpoint_scores.json'
if scores_checkpoint.exists():
    with open(scores_checkpoint, encoding='utf-8') as f:
        all_scores = json.load(f)
    print(f'Resuming: {len(all_scores)} questions judged')

for qi, q in enumerate(tqdm(questions, desc='Judging answers')):
    qid = q['qid']
    cik = q['cik']
    cd = company_data[cik]

    if qid in all_scores and len(all_scores[qid]) >= len(ALL_SYSTEMS):
        continue
    if qid not in all_scores:
        all_scores[qid] = {}

    for sys_name in ALL_SYSTEMS:
        if sys_name in all_scores[qid]: continue
        if sys_name == 'Full MSG-KG Framework' and 'MSG-KG Reasoning' in all_scores[qid]:
            all_scores[qid][sys_name] = all_scores[qid]['MSG-KG Reasoning']
            continue
        answer = all_answers.get(qid, {}).get(sys_name, '')
        if not answer:
            all_scores[qid][sys_name] = {'grounding': 1, 'explanation': 1}
            continue
        g_score, e_score = judge_answer(q, answer, cd['mission'], cd['kg_struct'])
        all_scores[qid][sys_name] = {'grounding': g_score, 'explanation': e_score}

    if (qi + 1) % 5 == 0:
        with open(scores_checkpoint, 'w', encoding='utf-8') as f:
            json.dump(all_scores, f, ensure_ascii=False)

with open(scores_checkpoint, 'w', encoding='utf-8') as f:
    json.dump(all_scores, f, ensure_ascii=False)
print('Judging complete.')

## Step 6: Compute Results — Tables 4 & 5

In [ ]:
# ── Compute and display Table 4 & 5 ──

def collect_scores(system_name, metric):
    scores = []
    for qid in all_scores:
        s = all_scores[qid].get(system_name, {}).get(metric, 2)
        scores.append(s)
    return normalize_score(scores)

# ── TABLE 4 ──
print('=' * 70)
print('TABLE 4: Evidence-Grounded Reasoning Performance')
print('=' * 70)
print(f'{"Model":<30} {"Grounding Acc":>15} {"Explanation Q":>15}')
print('-' * 62)

table4_data = {}
for sys_name in SYSTEMS:
    g_scores = collect_scores(sys_name, 'grounding')
    e_scores = collect_scores(sys_name, 'explanation')
    g_mean, g_lo, g_hi = bootstrap_ci(g_scores)
    e_mean, e_lo, e_hi = bootstrap_ci(e_scores)
    table4_data[sys_name] = {
        'grounding_mean': g_mean, 'grounding_ci': (g_lo, g_hi),
        'explanation_mean': e_mean, 'explanation_ci': (e_lo, e_hi),
        'grounding_scores': g_scores, 'explanation_scores': e_scores,
    }
    g_str = f'{g_mean:.1f} ({g_lo:.1f}-{g_hi:.1f})'
    e_str = f'{e_mean:.1f} ({e_lo:.1f}-{e_hi:.1f})'
    bold = '>>>' if sys_name == 'MSG-KG Reasoning' else '   '
    print(f'{bold}{sys_name:<27} {g_str:>15} {e_str:>15}')

# Significance tests
print('\nSignificance tests (MSG-KG vs baselines, paired bootstrap):')
msgkg_g = table4_data['MSG-KG Reasoning']['grounding_scores']
msgkg_e = table4_data['MSG-KG Reasoning']['explanation_scores']
for sys_name in SYSTEMS:
    if sys_name == 'MSG-KG Reasoning': continue
    p_g = paired_bootstrap_test(msgkg_g, table4_data[sys_name]['grounding_scores'])
    p_e = paired_bootstrap_test(msgkg_e, table4_data[sys_name]['explanation_scores'])
    sig_g = '*' if p_g < 0.05 else ''
    sig_e = '*' if p_e < 0.05 else ''
    print(f'  vs {sys_name:<28} Grounding p={p_g:.3f}{sig_g}  Explanation p={p_e:.3f}{sig_e}')

# ── TABLE 5 ──
print('\n' + '=' * 70)
print('TABLE 5: Ablation Study')
print('=' * 70)
print(f'{"Model Variant":<30} {"Grounding Acc":>15}')
print('-' * 47)

table5_data = {}
for variant_name in ABLATIONS:
    g_scores = collect_scores(variant_name, 'grounding')
    g_mean, g_lo, g_hi = bootstrap_ci(g_scores)
    table5_data[variant_name] = {'grounding_mean': g_mean, 'grounding_ci': (g_lo, g_hi), 'grounding_scores': g_scores}
    g_str = f'{g_mean:.1f} ({g_lo:.1f}-{g_hi:.1f})'
    print(f'{variant_name:<30} {g_str:>15}')

In [ ]:
# ── Save all results + generate .docx ──

def clean_for_json(obj):
    if isinstance(obj, (np.floating, float)): return float(obj)
    if isinstance(obj, (np.integer, int)): return int(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, dict): return {k: clean_for_json(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [clean_for_json(i) for i in obj]
    return obj

results = {
    'table4': clean_for_json(table4_data),
    'table5': clean_for_json(table5_data),
    'config': {
        'generator': GENERATOR_MODEL, 'judge': JUDGE_MODEL,
        'n_questions': len(questions), 'n_companies': len(registry),
        'n_bootstrap': N_BOOTSTRAP, 'confidence': CONFIDENCE,
    },
}

with open(RESULTS_DIR / 'full_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)
with open(RESULTS_DIR / 'all_answers.json', 'w', encoding='utf-8') as f:
    json.dump(all_answers, f, ensure_ascii=False)
with open(RESULTS_DIR / 'all_scores.json', 'w', encoding='utf-8') as f:
    json.dump(clean_for_json(all_scores), f, indent=2)

# Generate .docx
from docx import Document
from docx.enum.table import WD_TABLE_ALIGNMENT

doc = Document()
doc.add_heading('EMNLP Evaluation: Evidence-Grounded Reasoning', level=1)

doc.add_heading('Evaluation Setup', level=2)
doc.add_paragraph(
    f'We evaluate the MSG-KG framework on {len(registry)} S&P companies with SEC 10-K filings. '
    f'{len(questions)} test questions (3 per company) across three types: factual, strategic, trace. '
    f'Generator: {GENERATOR_MODEL} | Judge: {JUDGE_MODEL} (different model families to avoid self-preference bias). '
    f'Bootstrap 95% CIs (n={N_BOOTSTRAP}). Paired bootstrap significance tests.'
)

doc.add_heading('Table 4: Evidence-Grounded Reasoning Performance', level=2)
t4 = doc.add_table(rows=6, cols=3)
t4.alignment = WD_TABLE_ALIGNMENT.CENTER
for i, h in enumerate(['Model', 'Grounding Accuracy', 'Explanation Quality']):
    t4.rows[0].cells[i].text = h
for ri, sys_name in enumerate(['Text-Only LLM', 'Vector Retrieval + LLM', 'SentenceTransformer + RAG', 'GraphRAG', 'MSG-KG Reasoning']):
    d = table4_data[sys_name]
    t4.rows[ri+1].cells[0].text = sys_name
    t4.rows[ri+1].cells[1].text = f"{d['grounding_mean']:.1f} ({d['grounding_ci'][0]:.1f}-{d['grounding_ci'][1]:.1f})"
    t4.rows[ri+1].cells[2].text = f"{d['explanation_mean']:.1f} ({d['explanation_ci'][0]:.1f}-{d['explanation_ci'][1]:.1f})"

doc.add_heading('Table 5: Ablation Study', level=2)
t5 = doc.add_table(rows=5, cols=2)
t5.alignment = WD_TABLE_ALIGNMENT.CENTER
t5.rows[0].cells[0].text = 'Model Variant'
t5.rows[0].cells[1].text = 'Grounding Accuracy'
for ri, variant in enumerate(['Full MSG-KG Framework', 'Without Graph Retrieval', 'Without Evidence Linking', 'Without KG Structure']):
    d = table5_data[variant]
    t5.rows[ri+1].cells[0].text = variant
    t5.rows[ri+1].cells[1].text = f"{d['grounding_mean']:.1f} ({d['grounding_ci'][0]:.1f}-{d['grounding_ci'][1]:.1f})"

doc_path = RESULTS_DIR / 'evaluation_methodology.docx'
doc.save(str(doc_path))
print(f'Results saved to {RESULTS_DIR}')
print(f'DOCX: {doc_path}')

In [ ]:
# ── Download results ──
from google.colab import files

# Zip results
!cd /content/msgkg_eval && zip -r /content/eval_results.zip eval_results/
files.download('/content/eval_results.zip')
print('Download complete!')